# Evolutionary Sparsity Demonstration

This notebook demonstrates how to load, calculate, and validate evolutionary sparsity using ESM2 (Evolutionary Scale Modeling) probability scoring.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

from src.sparsity.evolutionary.evolutionary_sparsity import EvolutionarySparsity
from src.sparsity.evolutionary.evolutionary_validation import EvolutionaryValidation
from src.sparsity.evolutionary.esm_probability_engine import ESMProbabilityEngine

## 1. Explore the ESM2 Probability Engine

We can instantiate the ESM2 Probability Engine, which automatically attempts to load the real Hugging Face model and gracefully falls back to a deterministic, chemically plausible simulation if the network or download is unavailable.

In [ ]:
engine = ESMProbabilityEngine()
print(f"Mock active: {engine.is_mock_active}")

### 1.1 Compute Scores for Sample Mutations
Let's compare conservative vs radical substitutions. Hydrophobic to hydrophobic changes (like `A -> V`) should have higher likelihood (lower sparsity) than hydrophobic to charged changes (like `A -> D`).

In [ ]:
sequence = "MAPAAAPAAAPAAAAGAAAA"
p_conservative = engine.score_mutation(sequence, 12, "A", "V", "pA")
p_radical = engine.score_mutation(sequence, 12, "A", "D", "pA")

print(f"Conservative Probability (A -> V): {p_conservative:.6f}")
print(f"Radical Probability (A -> D):      {p_radical:.6f}")
print(f"P(conservative) > P(radical):       {p_conservative > p_radical}")

## 2. Run the Evolutionary Sparsity Pipeline

We can trigger the full pipeline to run on the registered datasets (`S001` and `S002`).

In [ ]:
pipeline = EvolutionarySparsity()
s_path, p_path, plot_path = pipeline.run_pipeline(
    sequence_path_override="data/raw/sequences/protein_sequences.parquet",
    mutations_path_override="data/raw/megascale_d/megascale_d.parquet",
    output_dir="results/evolutionary/"
)

print(f"Core output Parquet: {s_path}")
print(f"Probabilities Parquet: {p_path}")
print(f"Distribution Plot:     {plot_path}")

## 3. Inspect Output Dataset
Let's load the generated Parquet file to verify the columns and records are correctly populated.

In [ ]:
df = pd.read_parquet(s_path)
df

## 4. Run Scientific Validation & Certification

We verify that the generated dataset meets all structural and biological properties defined in the system specifications.

In [ ]:
validator = EvolutionaryValidation()
certified = validator.validate_dataset(s_path)
print(f"Evolutionary Sparsity Certified: {certified}")